# AdaptiveHb — Real Experiment (PyTorch backbones)

This is the notebook referenced by `PROJECT_MANIFEST.yaml`. It runs a genuine **baseline-vs-adaptive** experiment with the real PyTorch backbones on a dataset, then shows the archived metrics, the comparison (with paired significance), the reproducibility provenance manifest, and the publication figures.

**Requirements:** Python 3.11+, and the optional ML stack (PyTorch, torchvision, numpy, matplotlib, …). A GPU is recommended but not required.

Everything is driven from configuration — no dataset names, paths, image sizes, or model names are hardcoded.

## ⚙️ Settings — EDIT ONLY THIS CELL

Everything you change lives in the next cell. Dataset paths are taken **independently**, exactly where they belong:

* **Segmentation** — a per-tissue on/off flag with its own image + mask folders, plus `SEG_RUN_ON` listing which tissues to train.
* **Prediction** — one image path *per side* (`LEFT_EYE`, `RIGHT_EYE`, `LEFT_NAIL`, …). The two sides of a tissue are pooled against the same patient's row in the labels file; `SAMPLING_MODE` chooses whether each image is its own data point (`extended`) or one image per patient (`single`).
* **Labels** — one Excel/CSV file and the columns you need. **BMI is computed automatically** from height and weight when it is missing.

Set your values, then **Run All**. These values are applied on top of `configs/*.yaml` at runtime — the YAML files on disk are never modified. Any path set to `None` is skipped, so a dataset that only covers some tissues just works.

In [ ]:
# ================================================================
# ⚙️  USER SETTINGS — the ONLY cell you need to edit.
# ----------------------------------------------------------------
# Independent dataset paths, taken directly here (nothing is
# hardcoded in source). These values OVERRIDE configs/*.yaml at
# runtime; the files on disk stay untouched. Set any path to None
# to disable that side / tissue / item.
# ================================================================

# --- DRY RUN ----------------------------------------------------
USE_SYNTHETIC = True      # True => ignore the paths below and fabricate data to test the pipeline

# ================================================================
# 1) SEGMENTATION — enable per tissue, give its image + mask dirs,
#    then list which tissues to actually train segmentation on.
# ================================================================
EYES_SEG          = True
EYE_SEG_IMAGES    = "/path/to/eye/seg/images"
EYE_SEG_MASKS     = "/path/to/eye/seg/masks"

NAIL_SEG          = False
NAIL_SEG_IMAGES   = None
NAIL_SEG_MASKS    = None

TONGUE_SEG        = False
TONGUE_SEG_IMAGES = None
TONGUE_SEG_MASKS  = None

PALM_SEG          = False
PALM_SEG_IMAGES   = None
PALM_SEG_MASKS    = None

SEG_RUN_ON = ["eye"]              # tissues to train segmentation on, e.g. ["eye", "palm", "nail"]

# ================================================================
# 2) PREDICTION — one image path PER SIDE. The two sides of a
#    tissue are pooled against the SAME patient's Hb row. A side set
#    to None is skipped; tongue is single-sided.
# ================================================================
LEFT_EYE   = "/path/to/left_eye"
RIGHT_EYE  = "/path/to/right_eye"

LEFT_PALM  = None
RIGHT_PALM = None

LEFT_NAIL  = None
RIGHT_NAIL = None

TONGUE     = None

PRED_RUN_ON = ["eye"]             # tissues to train prediction on, e.g. ["eye", "palm", "nail"]

# How to correlate a tissue's images with one patient's single label:
#   "extended" => each eye/palm/nail image is its OWN data point (more data; agentic)
#   "single"   => one representative image per patient per tissue
SAMPLING_MODE = "extended"

# ================================================================
# 3) LABELS — ONE Excel or CSV file + the column names you need.
#    BMI is computed algorithmically from height/weight when blank.
# ================================================================
METADATA_FILE     = "/path/to/patients.xlsx"   # .xlsx / .xls / .csv
DATASET_ROOT      = None       # folder to resolve a RELATIVE metadata path; None => METADATA_FILE's folder

PATIENT_ID_COLUMN = "patient_id"
TARGET_COLUMN     = "hemoglobin"
HEIGHT_COLUMN     = "height"
WEIGHT_COLUMN     = "weight"
BMI_COLUMN        = "BMI"
# Columns carried through the pipeline (BMI included; auto-computed if absent).
METADATA_COLUMNS  = ["patient_id", "hemoglobin", "height", "weight", "BMI"]

# ================================================================
# 4) RUN OUTPUT + MODELS
#    Leave the model settings as None: don't switch models — let
#    the configured defaults (Fable-tuned) handle it.
# ================================================================
BASE_DIR        = None            # where run outputs go; None => <repo>/runs
EXPERIMENT_NAME = "real_run"
EPOCHS          = 10              # start small (3-5) to verify, then raise

SEG_MODELS         = None         # None keeps the config default
PRED_BACKBONES     = None
PRED_DEFAULT       = None
PRED_TISSUE_MODELS = None

print("User settings loaded — edit only this cell; the rest of the notebook reads from here.")


## Get the code (Kaggle / Colab / local)

Run the cell below **first**. On Kaggle or Colab it clones the repository and enters it; when you already run from inside a local clone it does nothing.

In [ ]:
# === Get the code — run this FIRST (Kaggle / Colab / local) ==================
import os, subprocess
from pathlib import Path

REPO_URL = "https://github.com/junaidmaqbool/AgenticHb.git"


def _has_repo(path) -> bool:
    return (Path(path) / "configs" / "project.yaml").is_file()


if _has_repo(Path.cwd()):
    pass                                        # already inside the repo (local run)
elif _has_repo(Path.cwd().parent):
    os.chdir(Path.cwd().parent)                 # notebook opened from notebooks/
elif Path("AgenticHb").exists() and _has_repo("AgenticHb"):
    # A previous clone exists — pull the latest so we never run stale code.
    subprocess.run(["git", "-C", "AgenticHb", "pull", "--ff-only"], check=False)
    os.chdir("AgenticHb")
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "AgenticHb"], check=True)
    os.chdir("AgenticHb")

print("Working directory:", Path.cwd())
assert _has_repo(Path.cwd()), "Repository not found - check the clone step above."

# Sanity check: confirm we have the up-to-date code (real PNG synthetic images).
_syn = (Path.cwd() / "src" / "adaptivehb" / "dataset" / "synthetic.py").read_text()
assert "_png_bytes" in _syn, (
    "Stale code detected. Delete the old clone and re-run:  !rm -rf AgenticHb"
)
print("Code is up to date.")


## 0. Install the framework with the ML extras

Run this **once**. From a local clone (recommended), start the notebook from the repo root so the editable install resolves:

```bash
pip install -e ".[ml]"
```

On Google Colab, clone first, then install:

```python
!git clone https://github.com/<your-org>/AgenticHb.git
%cd AgenticHb
%pip install -e ".[ml]"
```

In [ ]:
# Uncomment to install (only needed once per environment).
# %pip install -e ".[ml]"

## 1. Setup and environment check

In [ ]:
# --- Make the framework importable and locate the repository root -------------
# Works whether you `pip install -e .` the package or just run from a clone.
import sys
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    """Walk upward from `start` (default: cwd) until a folder with configs/project.yaml."""
    start = (start or Path.cwd()).resolve()
    for directory in (start, *start.parents):
        if (directory / "configs" / "project.yaml").is_file():
            return directory
    raise FileNotFoundError(
        "Could not locate the repository root (a folder with configs/project.yaml). "
        "Run this notebook from inside the AgenticHb repository."
    )


REPO_ROOT = find_repo_root()
SRC = REPO_ROOT / "src"
if SRC.is_dir() and str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))  # allows running without `pip install`

CONFIG_DIR = REPO_ROOT / "configs"
print("Repository root :", REPO_ROOT)
print("Config directory:", CONFIG_DIR)


In [ ]:
# Confirm the ML stack is present (this notebook needs it to train real backbones).
try:
    import torch
    print('torch            :', torch.__version__)
    print('CUDA available   :', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('CUDA device      :', torch.cuda.get_device_name(0))
except ImportError:
    print('PyTorch is NOT installed. Run the install cell in section 0 first.')
    print('(Without torch the pipeline still runs, but on reference models — use '
          'smoke_synthetic.ipynb for that.)')

## 2. Apply settings to the config

Reads the values from the **⚙️ Settings** cell and overlays them on the YAML config at runtime (files in `configs/` are left untouched). It builds the per-side `tissue_sources`, wires each tissue's segmentation masks, sets the sampling mode, and points the metadata loader at your Excel/CSV file (BMI is derived on load). You normally don't edit this cell — change everything up top.

In [ ]:
# ============================================================
# APPLY SETTINGS — no need to edit. Overlays the USER SETTINGS
# onto the YAML config at runtime (the configs/ folder is left
# untouched; nothing is committed back to git).
# ============================================================
from pathlib import Path
from adaptivehb.config import ConfigLoader
from adaptivehb.dataset import generate_synthetic_dataset

if BASE_DIR is None:
    BASE_DIR = str(REPO_ROOT / "runs")

IMAGES_SUBDIR, MASKS_SUBDIR = "images", "masks"

# Per-side PREDICTION image paths, grouped per tissue.
_PRED_SIDES = {
    "eye":    {"left": LEFT_EYE,  "right": RIGHT_EYE},
    "palm":   {"left": LEFT_PALM, "right": RIGHT_PALM},
    "nail":   {"left": LEFT_NAIL, "right": RIGHT_NAIL},
    "tongue": {"center": TONGUE},
}
# Per-tissue SEGMENTATION source (images + masks) and its enable flag.
_SEG = {
    "eye":    {"on": EYES_SEG   and "eye"    in SEG_RUN_ON, "images": EYE_SEG_IMAGES,    "masks": EYE_SEG_MASKS},
    "nail":   {"on": NAIL_SEG   and "nail"   in SEG_RUN_ON, "images": NAIL_SEG_IMAGES,   "masks": NAIL_SEG_MASKS},
    "tongue": {"on": TONGUE_SEG and "tongue" in SEG_RUN_ON, "images": TONGUE_SEG_IMAGES, "masks": TONGUE_SEG_MASKS},
    "palm":   {"on": PALM_SEG   and "palm"   in SEG_RUN_ON, "images": PALM_SEG_IMAGES,   "masks": PALM_SEG_MASKS},
}


def _build_tissue_sources():
    """Assemble dataset.tissue_sources from the per-side + segmentation paths.

    Each tissue uses its per-side prediction images (attaching the tissue's
    segmentation masks when segmentation is enabled for it). A tissue with no
    prediction sides but WITH segmentation enabled falls back to its seg
    image/mask dirs, so a segmentation-only tissue still yields samples.
    """
    sources, tissues = {}, []
    considered = set(PRED_RUN_ON) | {t for t, s in _SEG.items() if s["on"]}
    for tissue, side_paths in _PRED_SIDES.items():
        if tissue not in considered:
            continue
        seg = _SEG.get(tissue, {})
        seg_masks = seg.get("masks") if seg.get("on") else None
        sides = {}
        for side, images in side_paths.items():
            if images:
                entry = {"images": images}
                if seg_masks:
                    entry["masks"] = seg_masks
                sides[side] = entry
        if sides:
            sources[tissue] = {"sides": sides}
            tissues.append(tissue)
        elif seg.get("on") and seg.get("images"):
            sources[tissue] = {"images": seg["images"], "masks": seg.get("masks")}
            tissues.append(tissue)
    return sources, tissues


# Load the YAML config, then overlay the settings (source configs untouched).
config = ConfigLoader(CONFIG_DIR).load()
ds   = config.section("dataset")["dataset"]
seg  = config.section("segmentation")["segmentation"]
pred = config.section("prediction")["prediction"]

if USE_SYNTHETIC:
    # Fabricate a small, decodable dataset covering every tissue you listed so
    # the whole pipeline can be dry-run without any real data.
    DATASET_ROOT = str(Path(BASE_DIR) / "synthetic_dataset")
    syn_tissues = list(dict.fromkeys(list(PRED_RUN_ON) + list(SEG_RUN_ON))) or ["eye"]
    generate_synthetic_dataset(DATASET_ROOT, num_patients=24, seed=7, tissues=syn_tissues)
    ds["images_dir"], ds["masks_dir"] = IMAGES_SUBDIR, MASKS_SUBDIR
    ds["metadata_file"] = "metadata/patients.csv"
    ds["tissues"] = syn_tissues
    ds["tissue_sources"] = {}                     # conventional layout under the synthetic root
    ds["sampling_mode"] = SAMPLING_MODE
    ds["metadata"] = {**ds.get("metadata", {}),
                      "patient_id_column": "Patient_ID", "target_column": "Hemoglobin",
                      "mandatory_columns": ["Patient_ID", "Hemoglobin"],
                      "needed_columns": ["Patient_ID", "Hemoglobin", "Age", "Gender",
                                         "Height", "Weight", "BMI"],
                      "height_column": "Height", "weight_column": "Weight",
                      "bmi_column": "BMI", "compute_bmi": True}
    print("Generated a synthetic dataset at:", DATASET_ROOT)
    print("Synthetic tissues :", syn_tissues)
else:
    if DATASET_ROOT is None:
        DATASET_ROOT = str(Path(METADATA_FILE).resolve().parent)
    tissue_sources, active_tissues = _build_tissue_sources()
    ds["metadata_file"] = METADATA_FILE
    ds["images_dir"], ds["masks_dir"] = IMAGES_SUBDIR, MASKS_SUBDIR
    ds["tissues"] = active_tissues
    ds["tissue_sources"] = tissue_sources
    ds["sampling_mode"] = SAMPLING_MODE
    ds["metadata"] = {**ds.get("metadata", {}),
                      "patient_id_column": PATIENT_ID_COLUMN, "target_column": TARGET_COLUMN,
                      "mandatory_columns": [PATIENT_ID_COLUMN, TARGET_COLUMN],
                      "needed_columns": list(METADATA_COLUMNS),
                      "height_column": HEIGHT_COLUMN, "weight_column": WEIGHT_COLUMN,
                      "bmi_column": BMI_COLUMN, "compute_bmi": True}
    print("Labels file    :", METADATA_FILE)
    print("Dataset root   :", DATASET_ROOT)
    print("Active tissues :", active_tissues or "(NONE — set at least one prediction side path!)")
    print("Sampling mode  :", SAMPLING_MODE)
    print("Segmentation on:", [t for t, s in _SEG.items() if s["on"]] or "(none)")

# Model overrides (None keeps the config default — do not switch models).
if SEG_MODELS is not None:
    seg["available_models"] = list(SEG_MODELS)
    seg["default_model"] = SEG_MODELS[0]
if PRED_BACKBONES is not None:
    pred["available_models"] = list(PRED_BACKBONES)
if PRED_DEFAULT is not None:
    pred["default_model"] = PRED_DEFAULT
if PRED_TISSUE_MODELS is not None:
    pred["tissue_models"] = dict(PRED_TISSUE_MODELS)

print("Output dir     :", BASE_DIR)
print("Segmentation   :", seg["available_models"], "(default:", seg["default_model"] + ")")
print("Prediction     :", pred["available_models"], "(default:", pred["default_model"] + ")")
print("Epochs         :", EPOCHS)


## 3. Run the experiment

This trains every segmentation and per-tissue prediction model, registers them with checkpoints, then compares the static baseline against the adaptive pipeline on the held-out test split and archives all outputs. Equivalent terminal command:

```bash
adaptivehb experiment --dataset-root <DATASET_ROOT> --base-dir runs --epochs 10 --name real_run
```

In [ ]:
from adaptivehb.pipeline import HbPipeline

# Build from the (overridden) config; base_dir/dataset_root come from the control panel.
pipeline = HbPipeline(config, base_dir=BASE_DIR, dataset_root=DATASET_ROOT)
result = pipeline.experiment(EXPERIMENT_NAME, epochs=EPOCHS)
print("Experiment id :", result.experiment_id)
print("Archived at   :", result.root)


## 4. Metrics (adaptive pipeline)

In [ ]:
import json

metrics = result.metrics
for key in ('mae', 'rmse', 'r2', 'pearson', 'spearman', 'mean_bias'):
    if key in metrics:
        print(f'{key:>10}: {metrics[key]:.4f}')

metrics_json = Path(result.root) / 'metrics' / 'adaptive_metrics.json'
print('\nFull metrics file:', metrics_json)

## 5. Baseline vs adaptive comparison (with paired significance)

In [ ]:
comparison = result.comparison
print(json.dumps({k: v for k, v in comparison.items() if k != 'significance'}, indent=2, default=str))

sig = comparison.get('significance')
if sig:
    print('\n-- paired significance --')
    print('n_pairs         :', sig['n_pairs'])
    print('mean |err| diff :', round(sig['mean_abs_error_diff'], 4))
    print('paired t-test p :', round(sig['paired_t_test']['p_value'], 6))
    print('wilcoxon p      :', round(sig['wilcoxon']['p_value'], 6))
    print("cohen's d       :", round(sig['cohens_d'], 4))
    ci = sig['bootstrap_ci']
    print('bootstrap 95% CI:', (round(ci['ci_lower'], 4), round(ci['ci_upper'], 4)))
    print('significant     :', sig['significant_at'])

## 6. Reproducibility provenance

In [ ]:
prov = result.provenance
print(json.dumps(prov, indent=2, default=str))

## 7. Publication figures

In [ ]:
from IPython.display import Image, display

figure_dir = Path(result.root) / 'figures'
pngs = sorted(figure_dir.glob('*.png')) if figure_dir.is_dir() else []
print('figures:', [p.name for p in pngs])
for png in pngs:
    display(Image(filename=str(png)))

## 8. Where everything is archived

In [ ]:
root = Path(result.root)
for sub in sorted(p for p in root.iterdir() if p.is_dir()):
    files = sorted(f.name for f in sub.rglob('*') if f.is_file())
    if files:
        print(f'{sub.name}/')
        for f in files:
            print('   ', f)

## Next steps

The archived experiment directory contains everything needed for the paper: the metric bundle, the baseline-vs-adaptive comparison with paired significance, the per-sample predictions, the figures (scatter, Bland-Altman, comparison), the reproducibility manifest, and a summary. Re-run with a larger `EPOCHS` and your real dataset to produce the final publication assets (Papers 1–4).